In [82]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [83]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

model = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash',
    api_key = api_key
)

In [84]:
class EValuationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback for the essay')
    score : int = Field(description='Score out of 10', ge=0, le=10)
    

In [85]:
structured_model = model.with_structured_output(EValuationSchema)

In [86]:
class UPSCState(TypedDict):

    essay: str
    language_feedback:str
    analysis_feedback:str
    cot_feedback:str
    overall_feedback:str
    avg_score = int
    individual_scores: Annotated[list[int], operator.add]

In [87]:
def evaluate_language(state:UPSCState):

    essay = state['essay']
    prompt = f'Evaluate the language quality of the following essat and provide a feedback and assign a score out of 10 \n {essay}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores':[output.score]}

In [88]:
def evaluate_analysis(state:UPSCState):

    essay = state['essay']
    prompt = f'Evaluate the depth of analysis quality of the following essat and provide a feedback and assign a score out of 10 \n {essay}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores':[output.score]}

In [89]:
def evaluate_thought(state:UPSCState):

    essay = state['essay']
    prompt = f'Evaluate the thought of the following essay  and provide a feedback and assign a score out of 10 \n {essay}'
    output = structured_model.invoke(prompt)

    return {'cot_feedback': output.feedback, 'individual_scores':[output.score]}

In [90]:
def final_evaluation(state:UPSCState):

    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n Depth of analysis feedback = {state["analysis_feedback"]} \n Clarity of thought feedback - {state["cot_feedback"]}'

    overall_feedback = model.invoke(prompt).content
    avg_score= sum(state['individual_scores'])/len(state["individual_scores"])

    return {'overall_feedback':overall_feedback, 'avg_score':avg_score}
    

In [91]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [92]:
essay = """# The Impact of Artificial Intelligence on Human Society

## Introduction

Artificial Intelligence (AI) is one of the most transformative technologies of the modern era. It refers to the ability of computer systems to perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, understanding language, recognizing images, and making decisions. AI has evolved from a theoretical concept in computer science into a practical technology that influences almost every aspect of our daily lives.

Today, artificial intelligence powers virtual assistants, recommendation systems, medical diagnostic tools, autonomous vehicles, fraud detection systems, and generative AI applications capable of producing text, images, music, and software code. Its rapid development presents enormous opportunities for economic growth, scientific discovery, and social progress. However, it also raises concerns about employment, privacy, misinformation, security, and the ethical use of technology.

Understanding the impact of AI on human society requires examining both its benefits and its limitations. Rather than viewing AI as either a perfect solution or an unavoidable threat, society must learn how to develop and use it responsibly.

## 1. The Evolution of Artificial Intelligence

The idea of creating machines capable of intelligent behavior has existed for centuries, but AI became a formal field of scientific research during the twentieth century. In 1950, British mathematician Alan Turing explored the question of whether machines could demonstrate intelligent behavior. In 1956, the Dartmouth Summer Research Project helped establish artificial intelligence as a distinct academic field.

Early AI systems relied heavily on predefined rules and symbolic reasoning. They could solve specific logical problems but struggled with complex real-world situations. As computing power increased and larger datasets became available, researchers developed machine learning techniques that enabled computers to identify patterns and improve their performance through data.

Deep learning, a branch of machine learning based on artificial neural networks, produced significant advances in image recognition, speech processing, and natural language understanding. More recently, large language models and generative AI systems have demonstrated the ability to produce human-like text, write code, summarize documents, and assist with creative tasks.

This evolution illustrates an important principle: AI capabilities depend not only on algorithms but also on data quality, computing infrastructure, scientific research, and the ability to evaluate system performance.

## 2. Artificial Intelligence in Education

Education is one of the sectors where AI can create significant improvements. Traditional education often follows a standardized approach, even though students differ in their learning speeds, interests, strengths, and weaknesses. AI-powered educational systems can help address these differences by adapting learning experiences to individual needs.

For example, an intelligent tutoring system can identify topics a student finds difficult and provide additional explanations or practice questions. AI tools can summarize lengthy educational material, explain complex scientific concepts in simpler language, generate quizzes, and provide feedback on written assignments.

Teachers can also benefit from AI through administrative assistance, lesson planning, and the creation of educational resources. These tools may allow educators to devote more time to mentoring students and developing critical thinking skills.

However, the use of AI in education also presents challenges. Students may become overly dependent on automated answers instead of developing their own reasoning abilities. AI-generated explanations can contain factual errors, and automated assessments may unfairly evaluate certain students.

Therefore, AI should complement teachers rather than replace the human relationships and guidance that are essential to education. Students should use AI as a learning partner, verify important information, and develop the ability to solve problems independently.

## 3. Artificial Intelligence in Healthcare

Healthcare is another field in which artificial intelligence has considerable potential. Medical professionals must analyze large amounts of information, including patient histories, laboratory results, medical images, and research publications. AI systems can help process this information and identify patterns that may be difficult to detect manually.

For example, AI models can assist qualified professionals in analyzing X-rays, CT scans, and other medical images. Predictive systems may help identify patients at higher risk of certain complications, while natural language processing can assist with organizing medical records and extracting relevant information.

AI can also accelerate pharmaceutical research by helping scientists analyze molecular structures, identify promising drug candidates, and prioritize experiments. These capabilities may reduce the time required for some stages of scientific investigation.

Despite these advantages, AI cannot guarantee accurate diagnoses or eliminate the need for medical expertise. Biased training data can produce unequal results across different populations, and an incorrect prediction may have serious consequences.

Patient privacy is another major concern because medical information is highly sensitive. Healthcare organizations must protect patient data, evaluate AI systems rigorously, and ensure that qualified professionals remain responsible for clinical decisions.

The most beneficial use of AI in healthcare is to support medical professionals with additional information and analytical capabilities while preserving human judgment, accountability, and patient trust.

## 4. Artificial Intelligence and Employment

The relationship between AI and employment is one of the most debated aspects of technological progress. Throughout history, technological innovations have changed the nature of work. Industrial machinery transformed manufacturing, computers automated calculations and recordkeeping, and the internet created entirely new industries.

AI is likely to continue this process by automating certain repetitive, predictable, and information-intensive tasks. In offices, AI tools can draft documents, classify information, summarize meetings, and assist with data analysis. In software development, they can generate code suggestions, explain errors, and help developers understand unfamiliar systems.

Automation can improve productivity and reduce the time required to complete routine tasks. However, workers whose responsibilities consist largely of automatable activities may face pressure to change their roles or acquire new skills.

At the same time, AI creates opportunities in areas such as machine learning engineering, AI application development, data engineering, AI security, model evaluation, and technology governance. It can also enable small businesses and individuals to perform tasks that previously required larger teams or specialized resources.

The long-term effects on employment will depend on how organizations adopt AI, whether new jobs emerge, how quickly workers can retrain, and how the benefits of productivity are distributed.

Consequently, education and professional development must emphasize adaptable skills. Critical thinking, communication, creativity, domain expertise, and the ability to work effectively with AI systems will remain important as job responsibilities evolve.

## 5. Artificial Intelligence in Business and the Economy

Businesses increasingly use AI to improve efficiency, understand customers, forecast demand, detect fraud, and support decision-making. By analyzing large datasets, AI systems can identify patterns that help organizations allocate resources more effectively.

Retail businesses, for example, can use AI to estimate product demand and manage inventory. Banks can apply machine learning to detect unusual transactions. Manufacturers can use predictive maintenance systems to identify signs of equipment failure before a breakdown occurs.

Generative AI has also made it easier for businesses to produce marketing materials, draft customer communications, develop software prototypes, and provide automated customer support.

These capabilities may increase productivity and encourage innovation, particularly for smaller organizations that previously lacked access to advanced analytical tools.

Nevertheless, adopting AI involves costs and risks. Businesses may need to invest in computing resources, specialized personnel, data management, cybersecurity, and employee training. Poorly designed AI systems can produce inaccurate recommendations or expose confidential information.

There is also a risk that the economic benefits of AI will become concentrated among a small number of companies with access to large datasets, computing infrastructure, and capital.

To achieve broad economic benefits, organizations must combine technological innovation with responsible management, fair competition, workforce development, and appropriate safeguards.

## 6. Artificial Intelligence and Everyday Life

AI has become part of everyday life, often operating behind the scenes. Search engines use machine learning to rank information. Streaming platforms recommend films and music. Navigation applications estimate travel times and suggest routes. Email services filter spam, while smartphones use AI for photography, speech recognition, and predictive text.

Generative AI applications extend these capabilities by allowing people to interact with computer systems through natural language. Users can ask questions, brainstorm ideas, translate documents, generate images, or receive help with programming.

These tools can make information more accessible and reduce the effort required for many routine activities. They can also help people who face language barriers or certain accessibility challenges.

However, convenience can encourage excessive dependence. Recommendation algorithms may reinforce existing preferences, reducing exposure to different viewpoints. Automated systems may also collect extensive behavioral data to personalize their services.

People should therefore understand how digital systems influence their decisions and pay attention to privacy settings, data permissions, and the reliability of AI-generated information.

Technology should expand human capabilities without eliminating the independent judgment that allows individuals to make informed choices.

## 7. Ethical Challenges of Artificial Intelligence

The development of artificial intelligence raises important ethical questions concerning fairness, transparency, accountability, privacy, and human autonomy.

One major concern is algorithmic bias. AI systems learn patterns from training data, and that data may reflect historical discrimination or unequal representation. As a result, an AI system used for recruitment, lending, or other consequential decisions may produce unfair outcomes for particular groups.

Another concern is transparency. Some advanced AI models are difficult to interpret, making it challenging to understand why a particular prediction or recommendation was produced. In high-stakes situations, organizations need ways to evaluate system behavior, document limitations, and investigate harmful outcomes.

Privacy is equally important. AI applications may process personal communications, photographs, financial records, or other sensitive information. Organizations must handle such information carefully and comply with applicable data protection requirements.

Accountability must also be clearly established. When an AI system produces a harmful result, responsibility cannot simply be assigned to the technology itself. Developers, deploying organizations, and relevant decision-makers must have appropriate responsibilities for testing, monitoring, and correcting failures.

Ethical AI requires more than technical accuracy. It requires designing systems that respect human rights, distribute benefits fairly, minimize foreseeable harm, and remain subject to meaningful oversight.

## 8. Artificial Intelligence, Misinformation, and Cybersecurity

AI can strengthen cybersecurity by identifying suspicious network activity, detecting unusual login patterns, and helping security teams investigate potential threats. Automated analysis can allow organizations to identify certain attacks more quickly than manual monitoring alone.

At the same time, AI can be misused to create convincing phishing messages, imitate voices, generate deceptive images, and produce misleading videos. Such capabilities make it increasingly difficult to determine whether some digital content is authentic.

AI systems can also introduce security vulnerabilities through unsafe integrations, excessive permissions, exposed credentials, or the processing of malicious instructions. Organizations must therefore evaluate both the security of the models they use and the applications built around them.

Effective defenses include secure software development, access controls, employee education, independent testing, incident response planning, and verification of sensitive requests through trusted channels.

Individuals also need stronger digital literacy. Before sharing a dramatic claim, transferring money, or responding to an urgent message, they should verify the source through independent and reliable channels.

AI itself cannot eliminate misinformation or cybercrime. Its benefits depend on combining technical safeguards with responsible behavior and effective institutions.

## 9. Artificial Intelligence and Scientific Research

Scientific research increasingly involves analyzing complex systems and enormous quantities of data. AI can help researchers identify patterns, generate hypotheses, simulate processes, and prioritize experiments.

In biology, AI systems can assist with analyzing proteins and other molecular structures. In climate science, machine learning can help process environmental observations and improve certain forecasting methods. In astronomy, AI can help researchers examine large collections of images and identify unusual objects.

These applications can accelerate particular stages of discovery, allowing scientists to focus more attention on interpreting results and designing experiments.

However, a prediction is not automatically a scientific explanation. AI-generated hypotheses must be tested, experimental results must be reproduced where possible, and conclusions must be evaluated against established scientific evidence.

The future of scientific AI will depend on collaboration between researchers, computational systems, and experimental methods. Human curiosity and rigorous verification remain essential even when machines can process information at extraordinary scale.

## 10. The Future of Artificial Intelligence

The future of AI will likely involve increasingly capable systems integrated into software, industrial processes, scientific instruments, and personal devices. AI agents may become better at carrying out multistep tasks, interacting with external tools, and coordinating different stages of a workflow.

Advances in multimodal AI may allow systems to process text, images, audio, and video together. Smaller models may enable more AI applications to run locally on computers and mobile devices, potentially improving responsiveness and giving users greater control over some data.

AI could also contribute to advances in medicine, materials science, energy management, and personalized education.

Nevertheless, technical progress does not guarantee that every application will be beneficial. More capable systems can introduce new reliability, security, and governance challenges. The cost of computing resources and the environmental impact of large-scale infrastructure also deserve attention.

Governments, research institutions, businesses, and civil society will need to establish appropriate rules and standards for the development and deployment of AI. These measures should protect people without unnecessarily preventing beneficial research and innovation.

The objective should not be to maximize automation at any cost. It should be to use AI to improve human well-being, strengthen scientific and economic capabilities, and solve meaningful problems.

## Conclusion

Artificial intelligence is transforming how people learn, work, communicate, conduct research, and make decisions. Its ability to analyze information, automate repetitive tasks, and generate useful outputs offers significant opportunities for improving productivity and addressing complex challenges.

However, AI also introduces serious concerns involving employment transitions, privacy, discrimination, cybersecurity, misinformation, and the concentration of economic power. These challenges cannot be solved through algorithms alone. They require responsible engineering, effective regulation, public awareness, and continued investment in education.

Ultimately, artificial intelligence should be understood as a powerful tool rather than a complete substitute for human judgment. Its impact will depend not only on what machines become capable of doing but also on the choices people make about how those capabilities are used.

A society that combines technological innovation with ethical responsibility, critical thinking, and human-centered decision-making will be better positioned to benefit from AI. The most important question is therefore not simply how intelligent machines can become, but how humanity can use them to create a more capable, fair, secure, and prosperous future."""

In [93]:
initial_state = {
    'essay' : essay
}

output_state = workflow.invoke(initial_state)
print(output_state)


{'essay': '# The Impact of Artificial Intelligence on Human Society\n\n## Introduction\n\nArtificial Intelligence (AI) is one of the most transformative technologies of the modern era. It refers to the ability of computer systems to perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, understanding language, recognizing images, and making decisions. AI has evolved from a theoretical concept in computer science into a practical technology that influences almost every aspect of our daily lives.\n\nToday, artificial intelligence powers virtual assistants, recommendation systems, medical diagnostic tools, autonomous vehicles, fraud detection systems, and generative AI applications capable of producing text, images, music, and software code. Its rapid development presents enormous opportunities for economic growth, scientific discovery, and social progress. However, it also raises concerns about employment, privacy, misinformation, security,